# 05. 2D CNN Modeling & Milestone Evaluation

## Overview
This final notebook transitions the Speech Emotion Recognition (SER) pipeline from 1D tabular baselines to a deep learning architecture trained on continuous 2D Log-Mel spectrograms that was generated in the previous phase ([`04_spectrogram_features.ipynb`](./04_spectrogram_features.ipynb)). 

We will train a lightweight 2D Convolutional Neural Network (CNN) across our speaker-partitioned evaluation harness and synthesize our final 3-way milestone comparison.

## Objectives
1. Load `04_spectrogram_tensors.npz`, reshape tensors into channel-first 4D arrays, and establish an explicit integer mapping for all 16 compound target classes (`F_angry`, `M_calm`, etc.).
2. As on the baseline LR and the SVC models, apply the standardized 5-fold `StratifiedGroupKFold` harness partitioned strictly by `actor_id` to prevent acoustic data leakage across speakers.
3. Train a compact Conv2D architecture with Batch Normalization, Dropout, and Global Average Pooling using Early Stopping.
4. **Final 3-Way Benchmark:** Compare out-of-fold metrics across all three project milestones:
   - **Baseline 1:** `DummyClassifier` (uniform/stratified chance)
   - **Baseline 2:** Tabular Linear Baseline (`LogisticRegression` + `StandardScaler` on 26-dim MFCC summary statistics)
   - **Milestone 3:** 2D CNN (Continuous Log-Mel Spectrogram representations)
5. Save the best model weights and class mapping to `models/` for future use

In [4]:
import os
import random
from pathlib import Path
from typing import Dict, List, Tuple

import keras
from keras import layers
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import LabelEncoder
import tensorflow as tf
from tqdm.auto import tqdm